# DNNLS Live Arena — Small-data MNIST: Fix the Training Recipe

This notebook is a **complete, working pipeline** for the *Small-data MNIST* challenge:
it downloads 1000 handwritten digits, trains an MLP, and packs `submission.zip`
(`submission.py` + `weights.pt`) for the Arena.

**It is deliberately bad at its job.** The training recipe is naive and the model barely learns.
Your task this week is *learning engineering*:

1. **Make it train.** Run the notebook as-is, submit, and look at the loss curve. Then find out why
   the loss hardly moves and fix the recipe.
2. **Make it generalise.** With only 1000 examples an MLP memorises the training set easily.
   The score is accuracy on **5000 hidden test digits**, and the projector shows your *training*
   accuracy next to it. A big gap means you are overfitting.

Workflow (same as last week): change something → run all cells → download `submission.zip` → upload it on
the Arena student page. Submit as often as you like; your best score counts.

> **Rule of the game:** train only on the 1000 digits in `mnist_train.csv`. The challenge is about the
> recipe, not about finding more data.


## 0. Get the public challenge file

`mnist_train.csv` is committed in the project repository, so Colab can download it straight from GitHub.
The Arena server also serves it, but a server on your lecturer's laptop (classroom Wi-Fi, `localhost`, ...)
is *not* reachable from Colab, which runs in Google's cloud.

Order of attempts: GitHub → Arena URL → manual upload with the file picker.


In [ ]:
DATA_BASE_URL = "https://raw.githubusercontent.com/aljiro/interactive_coding/main/examples/data"   # public challenge files in the repository
ARENA_URL = "http://localhost:8000"  # address shown on the projector (only reachable from Colab if deployed on a public VM)
CHALLENGE = "mnist_small"
FILES = ["mnist_train.csv"]

import os, requests

def fetch(base_url):
    for name in FILES:
        r = requests.get(f"{base_url}/{name}", timeout=30)
        r.raise_for_status()
        if not r.text.startswith("id,"):
            raise ValueError(f"{name}: unexpected content")
        open(name, "wb").write(r.content)
        print("downloaded", name, len(r.content), "bytes")

try:
    fetch(DATA_BASE_URL)
except Exception as exc:
    print("GitHub download failed:", exc)
    try:
        fetch(f"{ARENA_URL}/api/challenges/{CHALLENGE}/resources")
    except Exception as exc2:
        print("Arena download failed:", exc2)
        try:
            from google.colab import files  # type: ignore
            print("\nUpload mnist_train.csv (download it from the Arena student page):")
            files.upload()
        except ImportError:
            pass

assert all(os.path.exists(f) for f in FILES), "missing challenge files"


## 1. Load the data

One row per digit: `id`, `label` (0–9) and 784 **raw pixel values in 0…255** (a 28 × 28 image, row by row).
The hidden test digits the server scores you on have exactly the same format.


In [ ]:
import numpy as np
import pandas as pd

train = pd.read_csv("mnist_train.csv")
X_train = train[[f"p{i}" for i in range(784)]].to_numpy(dtype=np.float32, copy=True)   # raw pixels, 0..255
y_train = train["label"].to_numpy(dtype=np.int64, copy=True)

print(X_train.shape, y_train.shape)
print("pixel range:", X_train.min(), "...", X_train.max())
print("digits per class:", np.bincount(y_train))


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 10, figsize=(10, 2.4))
for c in range(10):
    for r, idx in enumerate(np.flatnonzero(y_train == c)[:2]):
        axes[r, c].imshow(X_train[idx].reshape(28, 28), cmap="gray_r")
        axes[r, c].set_axis_off()
    axes[0, c].set_title(str(c))
plt.suptitle("Small-data MNIST – two training examples per class")
plt.show()


## 2. The model, as a Python bundle

The server unpacks `submission.zip` in an isolated container (no network, 1 CPU, 512 MB, 30 s) and calls

```python
def predict(X):   # X: numpy (N, 784) raw pixels 0..255  ->  numpy (N, 10) class probabilities
```

on the hidden test digits **and** on the public training digits (that is how the projector knows your
training accuracy). `X` arrives *raw*, exactly like `mnist_train.csv`, so whatever preprocessing you train
with has to be applied inside `predict` as well. That is why `preprocess()` lives in `submission.py` and is
used by both the training loop below and by `predict`.

Everything `predict` needs (architecture, weight loading) must live in `submission.py` and load files
**relative to `__file__`**.


In [ ]:
%%writefile submission.py
import os
import numpy as np
import torch
import torch.nn as nn

HERE = os.path.dirname(os.path.abspath(__file__))
N_CLASSES = 10


def preprocess(X):
    """Raw pixels (N, 784) in 0..255 -> model input. Used for training AND inside predict()."""
    return np.array(X, dtype=np.float32)      # a writable float32 copy


class MLP(nn.Module):
    def __init__(self, hidden=(256, 256, 256)):
        super().__init__()
        layers, d = [], 784
        for h in hidden:
            layers += [nn.Linear(d, h), nn.Sigmoid()]
            d = h
        layers.append(nn.Linear(d, N_CLASSES))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)          # logits (N, 10)


_model = None


def _load():
    global _model
    if _model is None:
        m = MLP()
        m.load_state_dict(torch.load(os.path.join(HERE, "weights.pt"), map_location="cpu"))
        m.eval()
        _model = m
    return _model


def predict(X):
    """X: numpy.ndarray (N, 784) raw pixels -> numpy.ndarray (N, 10) with class probabilities."""
    model = _load()
    with torch.no_grad():
        logits = model(torch.as_tensor(preprocess(X)))
        return torch.softmax(logits, dim=1).numpy().astype(np.float64)


## 3. Train

The baseline recipe: plain gradient descent on the whole training set at once, a small constant learning
rate, a fixed number of epochs, and nothing held out. It records the training loss and accuracy per epoch
so you can *look* at what is happening — make a habit of that before changing anything.


In [ ]:
import importlib, time, torch, torch.nn.functional as F
import submission
importlib.reload(submission)

torch.manual_seed(0)
model = submission.MLP()
opt = torch.optim.SGD(model.parameters(), lr=0.01)

X_all = torch.as_tensor(submission.preprocess(X_train))
y_all = torch.as_tensor(y_train)

EPOCHS = 300
BATCH_SIZE = len(X_all)          # full batch: one gradient step per epoch
history = {"loss": [], "train_acc": []}

t0 = time.time()
for epoch in range(EPOCHS):
    model.train()
    perm = torch.randperm(len(X_all))
    total_loss = 0.0
    for i in range(0, len(X_all), BATCH_SIZE):
        idx = perm[i:i + BATCH_SIZE]
        logits = model(X_all[idx])
        loss = F.cross_entropy(logits, y_all[idx])
        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item() * len(idx)
    model.eval()
    with torch.no_grad():
        train_acc = (model(X_all).argmax(1) == y_all).float().mean().item()
    history["loss"].append(total_loss / len(X_all))
    history["train_acc"].append(train_acc)
    if epoch % 50 == 0 or epoch == EPOCHS - 1:
        print(f"epoch {epoch:4d}  loss {history['loss'][-1]:.4f}  train acc {train_acc:.3f}")

print(f"trained in {time.time() - t0:.1f}s")
torch.save(model.state_dict(), "weights.pt")
print("saved weights.pt")


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(history["loss"]); ax[0].axhline(np.log(10), ls="--", c="gray", lw=1)
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("training loss"); ax[0].set_title("loss  (dashed = ln 10, i.e. guessing)")
ax[1].plot(history["train_acc"]); ax[1].set_ylim(0, 1)
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("training accuracy"); ax[1].set_title("training accuracy")
plt.tight_layout(); plt.show()

print(f"final training accuracy: {history['train_acc'][-1]:.3f}")
print("You have NO estimate of the accuracy on unseen digits yet. The hidden test score is what counts.")


## 4. Pack and download the submission

The sanity check below runs `predict` exactly as the server will: raw pixels in, an `(N, 10)` matrix of
probabilities out, every row summing to 1.


In [ ]:
import zipfile
importlib.reload(submission)
p = submission.predict(X_train[:5])
print("predict() ->", p.shape, "argmax:", p.argmax(1), "true:", y_train[:5])
assert p.shape == (5, 10), "predict must return (N, 10)"
assert np.allclose(p.sum(1), 1.0, atol=1e-4) and p.min() >= 0, "rows must be probabilities (softmax)"

with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("submission.py")
    zf.write("weights.pt")
print("submission.zip:", [i.filename for i in zipfile.ZipFile("submission.zip").infolist()])

try:
    from google.colab import files  # type: ignore
    files.download("submission.zip")
except ImportError:
    print("Not in Colab: submission.zip is in the current directory.")


## 5. Where to look

Submit the baseline first so you have a reference point on the projector. Then work through the recipe.
These are the knobs from this week's lecture; the order is roughly the order in which they usually matter.

**Make it train**
* *What does the network see?* Look at the input range in section 1 and at `preprocess()`.
* *Is the signal getting through?* Activation function, depth, initialisation. A loss that sits at ln 10 ≈ 2.30
  means the network is still guessing uniformly.
* *How are you stepping?* Optimiser, learning rate, momentum, batch size (how many gradient steps per epoch
  do you actually take?), number of epochs. Try a learning-rate schedule once things move.

**Make it generalise**
* *Measure it.* Hold out part of the training data as a validation set and track validation accuracy per
  epoch. If you do not measure generalisation, you cannot engineer it — and the projector will.
* *Stop in time.* Early stopping on the validation curve.
* *Shrink the hypothesis.* Weight decay, dropout, a smaller network.
* *More data without more data.* Data augmentation: small random shifts of the 28 × 28 image are very
  effective on digits (`torch.roll`, or a padding + random crop). Remember that `predict` still receives raw,
  unaugmented pixels.
* *Collapse.* If the loss explodes to `nan` or jumps back to 2.30, the learning rate is too high for the
  current recipe, or the activations died. Lower it, clip gradients, or check your initialisation.

Keep `preprocess()` and `predict()` consistent with the training loop: the server only ever sees `submission.py`.
